<h1 style="text-align: center;">Named Entity Recognition</h1>

The goal of this project is to build a Named Entity Recognition (NER) system that can identify important entities in text, such as persons, organizations, locations, and dates.

Import libraries and load data

In [4]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
#load data
df = pd.read_csv("ner_dataset.csv", encoding="latin1")

In [11]:
df.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


EDA - Exploratory Data Analysis

In [16]:
df.shape

(1048575, 4)

In [17]:
df.columns

Index(['Sentence #', 'Word', 'POS', 'Tag'], dtype='object')

In [18]:
df.isnull().sum()

,0
Sentence #,1000616
Word,10
POS,0
Tag,0


In [19]:
#fill missing sentence values forward
df["Sentence #"] = df["Sentence #"].fillna(method="ffill")

/tmp/ipykernel_245/3588955224.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Sentence #"] = df["Sentence #"].fillna(method="ffill")


In [20]:
#drop missing words
df = df.dropna(subset=["Word"])

In [21]:
#group words into sentences
sentences = df.groupby("Sentence #").apply(
    lambda s: [(w, t) for w, t in zip(s["Word"], s["Tag"])]
)

#Convert to list
sentences = list(sentences)

print("Number of sentences:", len(sentences))
print(sentences[0])

Number of sentences: 47959
[('Thousands', 'O'), ('of', 'O'), ('demonstrators', 'O'), ('have', 'O'), ('marched', 'O'), ('through', 'O'), ('London', 'B-geo'), ('to', 'O'), ('protest', 'O'), ('the', 'O'), ('war', 'O'), ('in', 'O'), ('Iraq', 'B-geo'), ('and', 'O'), ('demand', 'O'), ('the', 'O'), ('withdrawal', 'O'), ('of', 'O'), ('British', 'B-gpe'), ('troops', 'O'), ('from', 'O'), ('that', 'O'), ('country', 'O'), ('.', 'O')]


/tmp/ipykernel_245/657948167.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby("Sentence #").apply(


In [22]:
#get unique tags
tags = df["Tag"].unique().tolist()

print("Number of tags:", len(tags))
print(tags)

Number of tags: 17
['O', 'B-geo', 'B-gpe', 'B-per', 'I-geo', 'B-org', 'I-org', 'B-tim', 'B-art', 'I-art', 'I-per', 'I-gpe', 'I-tim', 'B-nat', 'B-eve', 'I-eve', 'I-nat']


Preprocessing

In [23]:
#create tag to index mapping
tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

tag2idx

{'O': 0,
 'B-geo': 1,
 'B-gpe': 2,
 'B-per': 3,
 'I-geo': 4,
 'B-org': 5,
 'I-org': 6,
 'B-tim': 7,
 'B-art': 8,
 'I-art': 9,
 'I-per': 10,
 'I-gpe': 11,
 'I-tim': 12,
 'B-nat': 13,
 'B-eve': 14,
 'I-eve': 15,
 'I-nat': 16}

In [24]:
#get all unique words
words = list(set(df["Word"].values))

print("Number of words:", len(words))

Number of words: 35177


In [25]:
#Create word to index mapping
word2idx = {w: i + 2 for i, w in enumerate(words)}

#Add special tokens
word2idx["PAD"] = 0
word2idx["UNK"] = 1

idx2word = {i: w for w, i in word2idx.items()}

print("Mapping created.")

Mapping created.


In [26]:
#convert sentences into numerical format
X = [[word2idx.get(w[0], 1) for w in s] for s in sentences]
y = [[tag2idx[w[1]] for w in s] for s in sentences]

print("Example encoded sentence:")
print(X[0])
print(y[0])

Example encoded sentence:
[26620, 33345, 30911, 19625, 12507, 4249, 8252, 12788, 16530, 26490, 19102, 30962, 18037, 21061, 843, 26490, 19513, 33345, 24421, 17272, 34286, 34675, 26640, 29981]
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0]


In [27]:
#check the maximum sentence length
max_len = max(len(s) for s in X)

print("Maximum sentence length:", max_len)

Maximum sentence length: 104


In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

#pad input sequences
X_padded = pad_sequences(
    sequences=X,
    maxlen=max_len,
    padding="post",
    value=word2idx["PAD"]
)

#Pad target sequences
y_padded = pad_sequences(
    sequences=y,
    maxlen=max_len,
    padding="post",
    value=tag2idx["O"]
)

print("X shape:", X_padded.shape)
print("y shape:", y_padded.shape)

X shape: (47959, 104)
y shape: (47959, 104)


In [29]:
from sklearn.model_selection import train_test_split

#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_padded,
    y_padded,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (38367, 104)
X_test shape: (9592, 104)
y_train shape: (38367, 104)
y_test shape: (9592, 104)


Modeling

In [31]:
#reshape target data for sequence labeling
y_train = np.expand_dims(y_train, axis=-1)
y_test = np.expand_dims(y_test, axis=-1)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

y_train shape: (38367, 104, 1, 1)
y_test shape: (9592, 104, 1, 1)


In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, TimeDistributed, Dense

#build a BiLSTM model for ner
model = Sequential([
    Embedding(input_dim=len(word2idx), output_dim=64, input_length=max_len),
    Bidirectional(LSTM(units=64, return_sequences=True)),
    TimeDistributed(Dense(len(tag2idx), activation="softmax"))
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
#Compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [37]:
#train the model
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=3,
    validation_split=0.1,
    verbose=1
)

Epoch 1/3
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 36s 28ms/step - accuracy: 0.9522 - loss: 0.1964 - val_accuracy: 0.9394 - val_loss: 0.0336
Epoch 2/3
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9382 - loss: 0.0268 - val_accuracy: 0.9382 - val_loss: 0.0248
Epoch 3/3
1080/1080 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9378 - loss: 0.0174 - val_accuracy: 0.9385 - val_loss: 0.0232


In [38]:
#Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9383 - loss: 0.0235
Test Accuracy: 0.9383222460746765


In [39]:
#predict tags for test sentences
y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=-1)

print("Prediction shape:", y_pred_labels.shape)

300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step
Prediction shape: (9592, 104)


In [40]:
#show one example with true and predicted tags
i = 0

for word_idx, true_tag_idx, pred_tag_idx in zip(X_test[i], y_test[i].flatten(), y_pred_labels[i]):
    if word_idx != word2idx["PAD"]:
        print(f"{idx2word[word_idx]:15} True: {idx2tag[true_tag_idx]:8} Pred: {idx2tag[pred_tag_idx]}")

The             True: O        Pred: O
report          True: O        Pred: O
calls           True: O        Pred: O
on              True: O        Pred: O
President       True: B-per    Pred: B-per
Bush            True: I-per    Pred: I-per
and             True: O        Pred: O
Congress        True: B-org    Pred: B-org
to              True: O        Pred: O
urge            True: O        Pred: O
Chinese         True: B-gpe    Pred: B-gpe
officials       True: O        Pred: O
not             True: O        Pred: O
to              True: O        Pred: O
use             True: O        Pred: O
the             True: O        Pred: O
global          True: O        Pred: O
war             True: O        Pred: O
against         True: O        Pred: O
terrorism       True: O        Pred: O
as              True: O        Pred: O
a               True: O        Pred: O
pretext         True: O        Pred: O
to              True: O        Pred: O
suppress        True: O        Pred: O
minoritie

Conclusion

The final model achieved strong performance in named entity recognition.

- Training Accuracy: ~94%
- Validation Accuracy: ~94%
- Test Accuracy: ~93.8%

Although accuracy can appear high due to many non-entity tokens ("O"), qualitative evaluation shows that the model successfully identifies key entities such as persons, organizations, and locations.

In [42]:
#Save the trained NER model
model.save("ner_bilstm_model.keras")

print("Model saved successfully.")

Model saved successfully.


In [43]:
import pickle

# Save preprocessing objects needed for inference
with open("word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

with open("idx2tag.pkl", "wb") as f:
    pickle.dump(idx2tag, f)

with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)

print("Mappings saved successfully.")

Mappings saved successfully.
